# Biohub Cell Tracking — PyTorch Training Notebook

This notebook trains the 3D U-Net model for cell detection and movement vector prediction on the zebrafish embryo cells dataset.

### Features:
- **Resume-aware**: Automatically searches for `last.pt`, `best.pt`, and `training_log.csv` in any attached dataset directories (using robust recursive search) and resumes seamlessly.
- **Log Continuity**: Appends training indicators to the loaded log history, ensuring history is not lost across run restarts.
- **Epoch-Level Outputs**: Saves `last.pt`, `best.pt`, and updates `training_log.csv` at the end of every epoch.
- **Live Plotting**: Generates and saves a training/validation curve graph (`loss_plot.png`) after every epoch and displays it in the notebook output.

In [1]:
# ── Install/upgrade dependencies ──
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

pip_install('zarr>=3.0', 'scipy>=1.10', 'pandas>=2.0', 'tqdm', 'pyyaml', 'matplotlib', 'scikit-image>=0.21')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 85.4 MB/s eta 0:00:00


In [2]:
# ── Environment & Path Setup ──
import os
import sys
import random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
from IPython.display import clear_output, display

# Ensure local src modules can be imported
if '/kaggle/working' not in sys.path:
    sys.path.insert(0, '/kaggle/working')

# Fast path resolution: Scan immediate subdirectories of /kaggle/input and skip the competition dataset
input_root = Path("/kaggle/input")
if input_root.exists():
    search_paths = [input_root]
    competitions_dir = input_root / "competitions"
    if competitions_dir.exists():
        search_paths.append(competitions_dir)
        
    for parent in search_paths:
        for child in parent.iterdir():
            if "biohub-cell-tracking-during-development" in child.name:
                continue  # Skip scanning the massive competition dataset to avoid timeouts
            if child.is_dir():
                if child.name == "src":
                    if str(child.parent) not in sys.path:
                        sys.path.insert(0, str(child.parent))
                        print(f"Added source directory to sys.path: {child.parent}")
                else:
                    for p in child.glob("src"):
                        if p.is_dir() and str(child) not in sys.path:
                            sys.path.insert(0, str(child))
                            print(f"Added source directory to sys.path: {child}")

# --- CONFIGURATION PATHS ---
# Directory to search for uploaded checkpoints (last.pt, best.pt, training_log.csv)
INPUT_CHECKPOINT_DIRS = [
    Path("/kaggle/input/datasets/freak2209/biohub-cell-tracking-src"),
    Path("/kaggle/working")
]

# Automatically resolve the competition dataset directory containing training Zarrs
COMPETITION_DATA_ROOT = None
if input_root.exists():
    search_dirs = [input_root]
    competitions_dir = input_root / "competitions"
    if competitions_dir.exists():
        search_dirs.append(competitions_dir)
        
    for parent in search_dirs:
        for child in parent.iterdir():
            if "biohub-cell-tracking-during-development" in child.name:
                candidates = []
                for pattern in ["*.zarr", "*/*.zarr", "*/*/*.zarr"]:
                    candidates.extend(list(child.glob(pattern)))
                if candidates:
                    train_candidates = [p for p in candidates if "train" in p.parts or "train" in str(p)]
                    if train_candidates:
                        COMPETITION_DATA_ROOT = str(train_candidates[0].parent)
                    else:
                        COMPETITION_DATA_ROOT = str(candidates[0].parent)
                    break

if COMPETITION_DATA_ROOT is None:
    COMPETITION_DATA_ROOT = "/kaggle/input/biohub-cell-tracking-during-development/train"

# Output directory for saving checkpoints and logs
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

last_ckpt_path = OUTPUT_DIR / "last.pt"
best_ckpt_path = OUTPUT_DIR / "best.pt"
log_csv_path = OUTPUT_DIR / "training_log.csv"
plot_png_path = OUTPUT_DIR / "loss_plot.png"

print(f"COMPETITION_DATA_ROOT auto-resolved to: {COMPETITION_DATA_ROOT}")
print("Paths configured.")

COMPETITION_DATA_ROOT auto-resolved to: /kaggle/input/competitions/biohub-cell-tracking-during-development/train
Paths configured.


In [3]:
# ── Config Setup ──
import sys
sys.path.insert(0, "/kaggle/input/datasets/freak2209/biohub-cell-tracking-src")

from src.config import Config, ModelConfig, DataConfig, TrainingConfig
from src.models.unet3d import CellTrackingUNet
from src.models.residual_unet3d import ResidualCellTrackingUNet

# Choose model architecture: CellTrackingUNet or ResidualCellTrackingUNet
MODEL_CLASS = ResidualCellTrackingUNet 

# Initialize configuration
config = Config(
    model=ModelConfig(
        in_channels=1,
        base_features=16,          # 16 for standard, 24 for high-capacity
        downsample_factors=[[1, 2, 2], [2, 2, 2], [2, 2, 2]],
        nms_kernel=3,
        nms_threshold=0.5
    ),
    data=DataConfig(
        data_root=COMPETITION_DATA_ROOT,
        patch_size=[64, 128, 128],  # [64, 128, 128] is mathematically valid for 3D tracking kernels for U-Net downsampling/decoding
        train_full_volume=False,
        gaussian_sigma=[1.0, 2.0, 2.0],
        cell_mask_radius=2,
        scale=[1.625, 0.40625, 0.40625],
        normalize_mode="per_volume",
        reject_empty=True,
        center_on_cell_prob=0.5,
        input_shape=[64, 256, 256]
    ),
    training=TrainingConfig(
        lr=1e-4,
        batch_size=2,
        epochs=15,
        amp=True,
        swa_start_epoch=80,
        val_every=500,
        w_bg=10.0,
        w_fg=1.0,
        lambda_mv=1.0,
        vector_mask_mode="full",
        vector_curriculum_epoch=50,
        checkpoint_dir=str(OUTPUT_DIR),
        num_workers=4,
        val_embryo_fraction=0.2,
        seed=42
    )
)

print("Config loaded successfully.")

Config loaded successfully.


In [4]:
# ── Checkpoint & Log History Resolution ──
import shutil

def find_file_fast(filename, folders, is_dir=False):
    resolved_folders = []
    for f in folders:
        resolved_folders.append(f)
        comp_dir = f / "competitions"
        if comp_dir.exists():
            resolved_folders.append(comp_dir)
            
    for folder in resolved_folders:
        if not folder.exists():
            continue
        if folder == Path("/kaggle/working"):
            target = folder / filename
            if is_dir and target.is_dir(): return target
            if not is_dir and target.is_file(): return target
        else:
            for child in folder.iterdir():
                if "biohub-cell-tracking-during-development" in child.name:
                    continue
                if is_dir and child.is_dir() and child.name == filename: return child
                if not is_dir and child.is_file() and child.name == filename: return child
                if child.is_dir():
                    for p in child.rglob(filename):
                        if is_dir and p.is_dir(): return p
                        if not is_dir and p.is_file(): return p
    return None

def reconstruct_pt(folder_path, target_path):
    print(f"\n[!] Found unzipped checkpoint folder at {folder_path}")
    print(f"    Re-zipping it back into {target_path}...")
    shutil.make_archive(str(target_path).replace('.pt', ''), 'zip', folder_path)
    if os.path.exists(str(target_path).replace('.pt', '.zip')):
        os.rename(str(target_path).replace('.pt', '.zip'), target_path)
    return target_path

# Check for unzipped Kaggle folders first
last_folder = find_file_fast("last", INPUT_CHECKPOINT_DIRS, is_dir=True)
best_folder = find_file_fast("best", INPUT_CHECKPOINT_DIRS, is_dir=True)

if last_folder and last_folder.is_dir() and not find_file_fast("last.pt", INPUT_CHECKPOINT_DIRS):
    input_last = reconstruct_pt(last_folder, OUTPUT_DIR / "last.pt")
else:
    input_last = find_file_fast("last.pt", INPUT_CHECKPOINT_DIRS)

if best_folder and best_folder.is_dir() and not find_file_fast("best.pt", INPUT_CHECKPOINT_DIRS):
    input_best = reconstruct_pt(best_folder, OUTPUT_DIR / "best.pt")
else:
    input_best = find_file_fast("best.pt", INPUT_CHECKPOINT_DIRS)

input_log = find_file_fast("training_log.csv", INPUT_CHECKPOINT_DIRS)

resume_ckpt = None
log_history = pd.DataFrame(columns=['epoch', 'train_loss', 'val_loss', 'val_loss_ci', 'val_loss_mv', 'lr'])
start_epoch = 0
best_val_loss = float('inf')
val_checks_without_improvement = 0

# Resolve last.pt
if input_last and input_last.exists():
    print(f"Found last.pt at: {input_last}")
    try:
        resume_ckpt = torch.load(input_last, map_location='cpu')
        val_checks_without_improvement = resume_ckpt.get('val_checks_without_improvement', 0)
        # USER REQUESTED LOGS DIRECTLY FROM THE LOADED CHECKPOINT!
        print(f"\n{'='*50}")
        print(f"✅ SUCCESS: Model successfully loaded from checkpoint!")
        print(f"   -> Checkpoint stopped at Epoch: {resume_ckpt.get('epoch', 'Unknown')}")
        if 'val_loss' in resume_ckpt:
            print(f"   -> Checkpoint recorded Val Loss: {resume_ckpt['val_loss']:.4f}")
        print(f"{'='*50}\n")
    except Exception as e:
        print(f"Failed to load last.pt: {e}")

# Resolve training_log.csv
if input_log and input_log.exists():
    print(f"Found training_log.csv at: {input_log}")
    try:
        log_history = pd.read_csv(input_log)
        if not log_history.empty and 'epoch' in log_history.columns:
            start_epoch = int(log_history['epoch'].max())
            print(f"Continuing from completed Epoch {start_epoch} (Starting Epoch {start_epoch + 1})")
            if 'val_loss' in log_history.columns:
                best_val_loss = log_history['val_loss'].min()
                print(f"Best recorded validation loss so far: {best_val_loss:.4f}")
    except Exception as e:
        print(f"Failed to load log history: {e}")

if start_epoch == 0:
    print("No previous checkpoints resolved. Starting fresh training from Epoch 1.")


Found training_log.csv at: /kaggle/input/datasets/freak2209/biohub-cell-tracking-src/src/training_log.csv
Continuing from completed Epoch 3 (Starting Epoch 4)
Best recorded validation loss so far: 0.0415


In [5]:
# ── Training Components Initialization ──
from src.train import set_seed, build_dataloaders, validate, _update_swa_bn
from src.losses.detection_loss import DetectionLoss

# Setup reproducibility
set_seed(config.training.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

# Ensure model input shape matches training patch size to avoid crop size mismatch
config.data.input_shape = config.data.patch_size

# Initialize Model
model = MODEL_CLASS.from_config(config).to(device)
print(f"Model Output Shape: {model.output_shape} (Input: {config.data.input_shape})")
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")

# Loss function
loss_fn = DetectionLoss(
    w_bg=config.training.w_bg,
    w_fg=config.training.w_fg,
    lambda_mv=config.training.lambda_mv,
    vector_mask_mode=config.training.vector_mask_mode,
    curriculum_epoch=config.training.vector_curriculum_epoch,
    output_shape=model.output_shape,
)

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=config.training.lr)

# Schedulers
n_non_swa_epochs = max(1, config.training.swa_start_epoch)
cosine_scheduler = CosineAnnealingLR(
    optimizer,
    T_max=n_non_swa_epochs,
    eta_min=config.training.lr * 0.01,
)

# SWA
swa_model = AveragedModel(model)
swa_scheduler = SWALR(optimizer, swa_lr=config.training.swa_lr if hasattr(config.training, 'swa_lr') else config.training.lr * 0.05)
swa_started = False

# AMP Scaler
use_amp = config.training.amp and device.type == "cuda"
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

# Dataloaders
train_loader, val_loader = build_dataloaders(config, model)

Training on: cuda
Model Output Shape: (16, 36, 36) (Input: [64, 128, 128])
Trainable parameters: 6,964,036
Dataset split: 128 train samples, 71 val samples


In [6]:
# ── Restore State dicts if Resuming ──
if resume_ckpt is not None:
    print("Restoring model state_dict...")
    model.load_state_dict(resume_ckpt['model'])
    
    print("Restoring optimizer state_dict...")
    optimizer.load_state_dict(resume_ckpt['optimizer'])
    
    if 'scheduler' in resume_ckpt and resume_ckpt['scheduler'] is not None:
        print("Restoring cosine scheduler state_dict...")
        cosine_scheduler.load_state_dict(resume_ckpt['scheduler'])
        
    if 'swa_scheduler' in resume_ckpt and resume_ckpt['swa_scheduler'] is not None:
        print("Restoring SWA scheduler state_dict...")
        swa_scheduler.load_state_dict(resume_ckpt['swa_scheduler'])
        
    if 'swa_started' in resume_ckpt:
        swa_started = resume_ckpt['swa_started']
        
    if 'swa_model' in resume_ckpt and resume_ckpt['swa_model'] is not None:
        print("Restoring SWA model state_dict...")
        swa_model.load_state_dict(resume_ckpt['swa_model'])
        
    if 'scaler' in resume_ckpt and resume_ckpt['scaler'] is not None:
        print("Restoring AMP scaler state_dict...")
        scaler.load_state_dict(resume_ckpt['scaler'])
        
    print("State restoration complete.")

In [7]:
# ── Helper: Live Plotting Function ──
def plot_loss_curves(log_df, save_path):
    if log_df.empty:
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Total loss
    axes[0, 0].plot(log_df['epoch'], log_df['train_loss'], label='Train Loss', marker='o')
    axes[0, 0].plot(log_df['epoch'], log_df['val_loss'], label='Val Loss', marker='x')
    axes[0, 0].set_title('Total Loss')
    axes[0, 0].legend()
    
    # CI loss
    axes[0, 1].plot(log_df['epoch'], log_df['train_loss_ci'], label='Train CI', marker='o')
    axes[0, 1].plot(log_df['epoch'], log_df['val_loss_ci'], label='Val CI', marker='x')
    axes[0, 1].set_title('Cell Indicator Loss')
    axes[0, 1].legend()
    
    # Tracking Losses
    axes[1, 0].plot(log_df['epoch'], log_df['train_loss_la'], label='Train LA (Full)', marker='o')
    axes[1, 0].plot(log_df['epoch'], log_df['train_loss_lb'], label='Train LB (Maxima)', marker='o')
    if 'val_loss_la' in log_df.columns:
        axes[1, 0].plot(log_df['epoch'], log_df['val_loss_la'], label='Val LA', marker='x')
        axes[1, 0].plot(log_df['epoch'], log_df['val_loss_lb'], label='Val LB', marker='x')
    axes[1, 0].set_title('Tracking Losses')
    axes[1, 0].legend()
    
    # Alpha
    axes[1, 1].plot(log_df['epoch'], log_df['train_alpha'], label='Curriculum Alpha', marker='o')
    axes[1, 1].set_title('Curriculum Alpha')
    axes[1, 1].legend()
    
    fig.tight_layout()  
    
    plt.savefig(save_path)
    display(fig)
    plt.close()


In [ ]:
# ── Main Training Loop ──
print(f"Beginning training run from Epoch {start_epoch + 1} to {config.training.epochs}...")

global_step = start_epoch * len(train_loader)

for epoch in range(start_epoch, config.training.epochs):
    loss_fn.set_epoch(epoch)
    model.train()

    if epoch >= config.training.swa_start_epoch and not swa_started:
        swa_started = True
        print(f"\nSWA activated at epoch {epoch + 1}")

    pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{config.training.epochs}")
    epoch_train_losses = {'loss': 0.0, 'loss_ci': 0.0, 'loss_mv': 0.0, 'loss_la': 0.0, 'loss_lb': 0.0, 'alpha': 0.0}
    n_batches = 0

    for batch in pbar:
        loss_fn.current_step = global_step
        volume = batch["volume"].to(device)
        targets = {
            "cell_indicator":   batch["cell_indicator"].to(device),
            "cell_mask":        batch["cell_mask"].to(device),
            "movement_vectors": batch["movement_vectors"].to(device),
        }

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            outputs = model(volume, inference=False)
            losses = loss_fn(outputs, targets)
            loss = losses["loss"]

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        scaler.step(optimizer)
        scaler.update()

        if swa_started:
            swa_model.update_parameters(model)

        for k in epoch_train_losses:
            if k in losses:
                epoch_train_losses[k] += float(losses[k])
        n_batches += 1
        global_step += 1

        pbar.set_postfix(
            loss=f"{losses['loss']:.4f}",
            ci=f"{losses['loss_ci']:.4f}",
            mv=f"{losses['loss_mv']:.4f}",
            lr=f"{optimizer.param_groups[0]['lr']:.2e}"
        )

    # Step lr scheduler
    if swa_started:
        swa_scheduler.step()
    else:
        cosine_scheduler.step()

    # End of epoch calculations
    avg_train = {k: v / max(n_batches, 1) for k, v in epoch_train_losses.items()}
    val_metrics = validate(model, loss_fn, val_loader, device)
    val_loss = val_metrics["loss"]
    val_loss_ci = val_metrics["loss_ci"]
    val_loss_mv = val_metrics["loss_mv"]
    val_loss_la = val_metrics.get("loss_la", 0.0)
    val_loss_lb = val_metrics.get("loss_lb", 0.0)
    val_alpha = val_metrics.get("alpha", 0.0)
    current_lr = optimizer.param_groups[0]['lr']
    
    print(f"\nEpoch {epoch + 1}: Train Loss = {avg_train['loss']:.4f} | Val Loss = {val_loss:.4f} (CI: {val_loss_ci:.4f}, MV: {val_loss_mv:.4f}, LA: {val_loss_la:.4f}, LB: {val_loss_lb:.4f}, Alpha: {val_alpha:.4f})")

    # Save best.pt if validation loss improved
    if val_loss < best_val_loss:
        val_checks_without_improvement = 0
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch + 1,
            'model': model.state_dict(),
            'val_loss': best_val_loss,
        }, best_ckpt_path, _use_new_zipfile_serialization=False)
        print(f"  ↑ New best checkpoint saved → {best_ckpt_path}")
    else:
        val_checks_without_improvement += 1
        print(f"  - No improvement for {val_checks_without_improvement} epochs")

    if config.training.patience > 0 and val_checks_without_improvement >= config.training.patience:
        print(f"Early stopping triggered.")
        break

    # Save last.pt after every epoch
    torch.save({
        'epoch': epoch + 1,
        'model': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'val_checks_without_improvement': val_checks_without_improvement,
        'scheduler': cosine_scheduler.state_dict(),
        'swa_scheduler': swa_scheduler.state_dict(),
        'swa_started': swa_started,
        'swa_model': swa_model.state_dict(),
        'scaler': scaler.state_dict(),
    }, last_ckpt_path, _use_new_zipfile_serialization=False)
    print(f"  Saved last checkpoint to {last_ckpt_path}")

    # Update log history and save CSV
    new_row = pd.DataFrame([{
        'epoch': epoch + 1,
        'train_loss': avg_train['loss'],
        'train_loss_ci': avg_train['loss_ci'],
        'train_loss_mv': avg_train['loss_mv'],
        'train_loss_la': avg_train['loss_la'],
        'train_loss_lb': avg_train['loss_lb'],
        'train_alpha': avg_train['alpha'],
        'val_loss': val_loss,
        'val_loss_ci': val_loss_ci,
        'val_loss_mv': val_loss_mv,
        'val_loss_la': val_loss_la,
        'val_loss_lb': val_loss_lb,
        'val_alpha': val_alpha,
        'lr': current_lr
    }])
    log_history = pd.concat([log_history, new_row], ignore_index=True)
    log_history.to_csv(log_csv_path, index=False)
    print(f"  Updated log file at {log_csv_path}")

    # Draw and display updated loss plot
    clear_output(wait=True)
    plot_loss_curves(log_history, plot_png_path)

# ---- Post-training SWA updates ----
if swa_started:
    print("Updating SWA batch norms...")
    _update_swa_bn(swa_model, train_loader, device)
    torch.save({
        'model': swa_model.module.state_dict(),
    }, OUTPUT_DIR / "swa_final.pt", _use_new_zipfile_serialization=False)
    print(f"Saved swa_final.pt to {OUTPUT_DIR}")

print("Training finished successfully!")


Beginning training run from Epoch 4 to 15...


Epoch 4/15:   0%|          | 0/6270 [00:00<?, ?it/s]

/tmp/ipykernel_58/4280769075.py:29: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):
/tmp/ipykernel_58/4280769075.py:47: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  epoch_train_losses[k] += float(losses[k])


In [ ]:
# import argparse
# import random
# import shutil
# from pathlib import Path

# def main():
#     parser = argparse.ArgumentParser(description="Create a random subset of the Biohub training data.")
#     parser.add_argument("--input_dir", type=str, default="/kaggle/input/competitions/biohub-cell-tracking-during-development/train", help="Path to the original train directory (containing .zarr folders)")
#     parser.add_argument("--output_dir", type=str, default="/kaggle/working/train_subset", help="Path to the output subset directory")
#     parser.add_argument("--num_samples", type=int, default=70, help="Number of samples to extract")
#     parser.add_argument("--seed", type=int, default=42, help="Random seed for selection")
#     args = parser.parse_args()
    
#     input_dir = Path(args.input_dir)
#     output_dir = Path(args.output_dir)
    
#     if not input_dir.exists():
#         print(f"Error: Input directory '{input_dir}' does not exist.")
#         print("Please point --input_dir to your actual Kaggle dataset directory.")
#         return
        
#     print(f"Scanning '{input_dir}' for .zarr directories...")
#     # Find all .zarr directories
#     zarr_paths = [p for p in input_dir.rglob("*.zarr") if p.is_dir()]
    
#     if not zarr_paths:
#         print(f"No .zarr directories found in {input_dir}.")
#         return
        
#     print(f"Found {len(zarr_paths)} .zarr directories.")
    
#     # Shuffle and select
#     random.seed(args.seed)
#     num_to_select = min(args.num_samples, len(zarr_paths))
#     selected = random.sample(zarr_paths, num_to_select)
    
#     print(f"Randomly selected {len(selected)} samples. Copying to '{output_dir}'...")
#     output_dir.mkdir(parents=True, exist_ok=True)
    
#     for i, zarr_path in enumerate(selected):
#         # We try to maintain the relative folder structure 
#         # (in case there are subfolders before the .zarr)
#         try:
#             rel_path = zarr_path.relative_to(input_dir)
#         except ValueError:
#             rel_path = Path(zarr_path.name)
            
#         dest_path = output_dir / rel_path
        
#         print(f"[{i+1}/{len(selected)}] Copying {zarr_path.name}...")
        
#         if dest_path.exists():
#             print(f"  -> {dest_path} already exists. Skipping.")
#         else:
#             dest_path.parent.mkdir(parents=True, exist_ok=True)
#             shutil.copytree(zarr_path, dest_path, dirs_exist_ok=True)
            
#     print(f"\nSuccess! Created a subset with {len(selected)} samples in '{output_dir}'.")
#     print(f"You can now point your config.data.data_root to '{output_dir}' to test locally.")

# if __name__ == "__main__":
#     main()
